# Constraint Satisfaction Problem

## Aufgabe: 
### Labor- und Präsentationsplanung als Constraint Satisfaction Problem (CSP)


In [11]:
%pip install python-constraint ortools pandas jinja2


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### CSP-Konfiguration laden und darstellen

Tragen Sie den Namen des Konfigurationsfiles ein. Dieses wird kurz analysiert und dargestellt. Danach können Sie es geeignet dem CSP übergeben.


In [18]:
# Ihr Dateiname - nacheinander mehrere Konfigurationen testen und analysieren.
from csp_utils import analyze_and_display
config_file = "configC.json"
analyze_and_display(config_file)


# CSP-Konfiguration

Verwendete Datei: /Users/maximtausch/Documents/Studium/6. Semester/Grundlagen der KI/Reichhardt/C1/configC.json
Lademodus: json


## Übersicht

Gruppen:
  G1, G2, G3, G4, G5, G6, G7, G8, G9
Tage:
  Mon, Tue, Wed, Thu, Fri
Zeitslots:
  1: 08:00–10:00
  2: 10:30–12:30
  3: 13:00–14:30
  4: 15:00–17:00
Räume:
  L1, L2, L3
Anzahl Kommissionen:
  3
Anzahl Verfügbarkeits-Einträge:
  45


## Kompetenzen der Kommissionen

...

## Legende

Kommission,Farbe
K1,
K2,
K3,


## Wochenplan der Verfügbarkeiten

,Mon,Tue,Wed,Thu,Fri
Zeitslot,,,,,
1 (08:00–10:00),K3,"K2, K3","K1, K2, K3","K1, K2, K3","K1, K2"
2 (10:30–12:30),K3,"K1, K2, K3","K1, K2, K3","K1, K2, K3","K1, K2"
3 (13:00–14:30),K3,"K1, K2, K3","K1, K2, K3","K2, K3","K1, K2"
4 (15:00–17:00),K3,"K1, K2, K3","K1, K2, K3","K2, K3","K1, K2"


{'groups': ['G1', 'G2', 'G3', 'G4', 'G5', 'G6', 'G7', 'G8', 'G9'],
 'days': ['Mon', 'Tue', 'Wed', 'Thu', 'Fri'],
 'timeslots': {'1': '08:00–10:00',
  '2': '10:30–12:30',
  '3': '13:00–14:30',
  '4': '15:00–17:00'},
 'rooms': ['L1', 'L2', 'L3'],
 'commissions': {'K1': ['A', 'B'], 'K2': ['A'], 'K3': ['A', 'C']},
 'availability': {'K1': [['Tue', 2],
   ['Tue', 3],
   ['Tue', 4],
   ['Wed', 1],
   ['Wed', 2],
   ['Wed', 3],
   ['Wed', 4],
   ['Thu', 1],
   ['Thu', 2],
   ['Fri', 1],
   ['Fri', 2],
   ['Fri', 3],
   ['Fri', 4]],
  'K2': [['Tue', 1],
   ['Tue', 2],
   ['Tue', 3],
   ['Tue', 4],
   ['Wed', 1],
   ['Wed', 2],
   ['Wed', 3],
   ['Wed', 4],
   ['Thu', 1],
   ['Thu', 2],
   ['Thu', 3],
   ['Thu', 4],
   ['Fri', 1],
   ['Fri', 2],
   ['Fri', 3],
   ['Fri', 4]],
  'K3': [['Mon', 1],
   ['Mon', 2],
   ['Mon', 3],
   ['Mon', 4],
   ['Tue', 1],
   ['Tue', 2],
   ['Tue', 3],
   ['Tue', 4],
   ['Wed', 1],
   ['Wed', 2],
   ['Wed', 3],
   ['Wed', 4],
   ['Thu', 1],
   ['Thu', 2],
   ['Th

## Definition des Constraint Problems

Sie können die Bibliothek aus der Vorlesung nutzen, aber auch andere, z.B. diese:

| Bibliothek            | Vorteile |
|----------------------|---------|
| python-constraint    | Sehr einfach zu nutzen, ideal für klassische CSPs, perfekt für einfache Einführungsaufgaben |
| Google OR-Tools      | Sehr leistungsfähig, unterstützt komplexe Optimierung und Scheduling, industrieller Standard |
| Pyomo                | Saubere mathematische Modellierung, gut für lineare und ganzzahlige Optimierung |
| MiniZinc             | Speziell für Constraint-Probleme entwickelt, klare und strukturierte Modellierung |

In [19]:
from constraint import Problem
from itertools import combinations
import json

with open(config_file, "r") as f:
    config = json.load(f)

len_days = len(config["days"])

len_timeslots = len(config["timeslots"])

DAYS_TO_INT = {"Mon": 0, "Tue": 1, "Wed": 2, "Thu": 3, "Fri": 4}


PRESENTATION_ROOMS = {"A": ["L1", "L2"], "B": ["L3"], "C": ["L1", "L2", "L3"]}


# The two hardest things in programming are naming things, cache invalidation and off-by-one errors.
dhbw = Problem()

# domain for the variable: The details for the slot a presentation is held in
slot_domains = [
    (day, slot, commission, room)
    for day in config["days"]
    for slot in config["timeslots"]
    for commission in config["commissions"]
    for room in config["rooms"]
]

# primary variable - pairs of groups and the presentation they are holding
presentations = [
    (group, presentation)
    for group in config["groups"]
    for presentation in ["A", "B", "C"]
]

for pres in presentations:
    dhbw.addVariable(pres, slot_domains)

## hard constraints
for pres in presentations:
    # 7. Präsentationen können nur in bestimmten Räumen gehalten werden
    def matches_room_constraint(topic):
        return lambda slot: slot[3] in PRESENTATION_ROOMS[topic]
    
    dhbw.addConstraint(
        matches_room_constraint(pres[1]),
        (pres,)
    )
    
    # 6. Präsentationen können nur von bestimmten Kommissionen betreut werden
    def matches_commission_constraint(topic):
        return lambda slot: topic in config["commissions"][slot[2]]
    
    dhbw.addConstraint(
        matches_commission_constraint(pres[1]),
        (pres,)
    )

    # 8. Kommissionen können nur an bestimmten Slots
    dhbw.addConstraint(
        lambda slot: [slot[0], int(slot[1])] in config["availability"][slot[2]],
        (pres,)
    )
    

for pres_a, pres_b in combinations(presentations, 2):

    # 1. Keine Kommission doppelt belegt
    # 3. Kein Raum doppelt belegt
    dhbw.addConstraint(
        lambda slot_a, slot_b: slot_a[0] != slot_b[0]
        or slot_a[1] != slot_b[1]
        or (slot_a[2] != slot_b[2] and slot_a[3] != slot_b[3]),
        (pres_a, pres_b),
    )
    

    # Constraints affecting only presentations held by the same group
    if pres_a[0] == pres_b[0]:

        # 9. Maximal eine Präsentation pro Tag pro Gruppe
        # (impliziert 2. Keine Projektgruppe zu einem Zeitpunkt doppelt belegt)
        dhbw.addConstraint(
            lambda slot_a, slot_b: slot_a[0] != slot_b[0], (pres_a, pres_b)
        )

        if pres_a[1] < pres_b[1]:
        # 4. Präsentationsreihenfolge A -> B -> C
            dhbw.addConstraint(
                lambda slot_a, slot_b: DAYS_TO_INT[slot_a[0]] < DAYS_TO_INT[slot_b[0]],
                (pres_a, pres_b),
            )

            # 5. Mindestabstand zwischen Präsentationen jeder Gruppe von 2 Zeitslots
            dhbw.addConstraint(
                lambda slot_a, slot_b: (int(slot_b[1]) + 4 * DAYS_TO_INT[slot_b[0]])
                - (int(slot_a[1]) + 4 * DAYS_TO_INT[slot_a[0]])
                > 2,
                (pres_a, pres_b),
            )

for pres_a, pres_b, pres_c, pres_d in combinations(presentations, 4):
    def ensure_lazy_commissions(slot_a, slot_b, slot_c, slot_d) -> bool:
        # Check if all are on the same day
        if pres_a[0] == pres_b[0] and pres_a[0] == pres_c[0] and pres_a[0] == pres_d[0]:
            # print('hellooooo')
            # sort slots by their time slot
            slots_sorted = sorted([slot_a, slot_b, slot_c, slot_d], key=lambda item: (item[1]))
            print(slots_sorted)
            room_switches = 0
            previous_room = slot_a[3]
            for sl in slots_sorted[1:]:
                if sl[3] != previous_room:
                    room_switches += 1
                    previous_room = sl[3]
            return room_switches <= 1
        return True
    
    dhbw.addConstraint(ensure_lazy_commissions, (pres_a, pres_b, pres_c, pres_d))

solution = dhbw.getSolution()
if solution:
    solution = dict(sorted(solution.items(), key=lambda item: (item[0][0], item[0][1])))
solution
# print(solutions)
# print(solutions.keys[0])

KeyboardInterrupt: 

In [20]:
# Alternative Modellierung mit Google OR-Tools (CP-SAT) inkl. Optimierung weicher Constraints
from ortools.sat.python import cp_model
import json

with open(config_file, "r") as f:
    config = json.load(f)

len_timeslots = len(config["timeslots"])

ordered_timeslots = sorted(config["timeslots"], key=lambda ts: int(ts))

DAYS_TO_INT = {"Mon": 0, "Tue": 1, "Wed": 2, "Thu": 3, "Fri": 4}

PRESENTATION_ROOMS = {"A": ["L1", "L2"], "B": ["L3"], "C": ["L1", "L2", "L3"]}

presentations = [
    (group, topic)
    for group in config["groups"]
    for topic in ["A", "B", "C"]
]

slots = [
    (day, slot, commission, room)
    for day in config["days"]
    for slot in ordered_timeslots
    for commission in config["commissions"]
    for room in config["rooms"]
]

def allowed_slot(presentation, slot):
    topic = presentation[1]
    day, timeslot, commission, room = slot
    return (
        room in PRESENTATION_ROOMS[topic]
        and topic in config["commissions"][commission]
        and [day, int(timeslot)] in config["availability"][commission]
    )

model = cp_model.CpModel()

# x[(p_idx, s_idx)] == 1, falls Presentation p_idx auf Slot s_idx gelegt wird
x = {}
for p_idx, pres in enumerate(presentations):
    for s_idx, slot in enumerate(slots):
        if allowed_slot(pres, slot):
            x[(p_idx, s_idx)] = model.NewBoolVar(f"x_{p_idx}_{s_idx}")

# jede Präsentation genau einem gültigen Slot zuweisen
for p_idx, _ in enumerate(presentations):
    valid_vars = [x[(p_idx, s_idx)] for s_idx in range(len(slots)) if (p_idx, s_idx) in x]
    model.Add(sum(valid_vars) == 1)

# Hilfs-Ausdrücke: gewählter Tag und absoluter Slotindex je Präsentation
day_expr = {}
abs_slot_expr = {}
for p_idx, _ in enumerate(presentations):
    day_terms = []
    abs_terms = []
    for s_idx, slot in enumerate(slots):
        if (p_idx, s_idx) not in x:
            continue
        day, timeslot, _, _ = slot
        day_idx = DAYS_TO_INT[day]
        abs_idx = day_idx * len_timeslots + int(timeslot)
        day_terms.append(day_idx * x[(p_idx, s_idx)])
        abs_terms.append(abs_idx * x[(p_idx, s_idx)])
    day_expr[p_idx] = sum(day_terms)
    abs_slot_expr[p_idx] = sum(abs_terms)

# 1. Keine Kommission doppelt belegt
for day in config["days"]:
    for timeslot in ordered_timeslots:
        for commission in config["commissions"]:
            concurrent = [
                x[(p_idx, s_idx)]
                for p_idx in range(len(presentations))
                for s_idx, slot in enumerate(slots)
                if (p_idx, s_idx) in x
                and slot[0] == day
                and slot[1] == timeslot
                and slot[2] == commission
            ]
            if concurrent:
                model.Add(sum(concurrent) <= 1)

# 3. Kein Raum doppelt belegt
for day in config["days"]:
    for timeslot in ordered_timeslots:
        for room in config["rooms"]:
            concurrent = [
                x[(p_idx, s_idx)]
                for p_idx in range(len(presentations))
                for s_idx, slot in enumerate(slots)
                if (p_idx, s_idx) in x
                and slot[0] == day
                and slot[1] == timeslot
                and slot[3] == room
            ]
            if concurrent:
                model.Add(sum(concurrent) <= 1)

# gruppenbezogene Hard Constraints
for group in config["groups"]:
    group_p_idxs = [i for i, (g, _) in enumerate(presentations) if g == group]

    # 9. Maximal eine Präsentation pro Tag pro Gruppe
    for day in config["days"]:
        same_day = [
            x[(p_idx, s_idx)]
            for p_idx in group_p_idxs
            for s_idx, slot in enumerate(slots)
            if (p_idx, s_idx) in x and slot[0] == day
        ]
        if same_day:
            model.Add(sum(same_day) <= 1)

    group_topics = {presentations[i][1]: i for i in group_p_idxs}

    # 4. Präsentationsreihenfolge A -> B -> C
    model.Add(day_expr[group_topics["A"]] < day_expr[group_topics["B"]])
    model.Add(day_expr[group_topics["B"]] < day_expr[group_topics["C"]])

    # 5. Mindestabstand > 2 Zeitslots
    model.Add(abs_slot_expr[group_topics["B"]] - abs_slot_expr[group_topics["A"]] > 2)
    model.Add(abs_slot_expr[group_topics["C"]] - abs_slot_expr[group_topics["B"]] > 2)

def build_occupancy(resource_idx, resources, prefix):
    """occupancy[(resource, day, timeslot)] = 1, falls Resource in diesem Slot belegt ist."""
    occupancy = {}
    for resource in resources:
        for day in config["days"]:
            for timeslot in ordered_timeslots:
                occ = model.NewBoolVar(f"occ_{prefix}_{resource}_{day}_{timeslot}")
                concurrent = [
                    x[(p_idx, s_idx)]
                    for p_idx in range(len(presentations))
                    for s_idx, slot in enumerate(slots)
                    if (p_idx, s_idx) in x
                    and slot[0] == day
                    and slot[1] == timeslot
                    and slot[resource_idx] == resource
                ]
                if concurrent:
                    model.Add(sum(concurrent) == occ)
                else:
                    model.Add(occ == 0)
                occupancy[(resource, day, timeslot)] = occ
    return occupancy

def add_internal_idle_penalties(occupancy, resources, prefix):
    """Bestrafe Leerlauf in einem Slot, wenn davor und danach am selben Tag Belegung existiert."""
    idle_vars = []
    n = len(ordered_timeslots)
    if n <= 2:
        return idle_vars

    for resource in resources:
        for day in config["days"]:
            has_before = {}
            has_after = {}

            first_ts = ordered_timeslots[0]
            first_before = model.NewBoolVar(f"before_{prefix}_{resource}_{day}_{first_ts}")
            model.Add(first_before == 0)
            has_before[first_ts] = first_before

            for i in range(1, n):
                ts = ordered_timeslots[i]
                prev_ts = ordered_timeslots[i - 1]
                b = model.NewBoolVar(f"before_{prefix}_{resource}_{day}_{ts}")
                model.Add(b >= has_before[prev_ts])
                model.Add(b >= occupancy[(resource, day, prev_ts)])
                model.Add(b <= has_before[prev_ts] + occupancy[(resource, day, prev_ts)])
                has_before[ts] = b

            last_ts = ordered_timeslots[-1]
            last_after = model.NewBoolVar(f"after_{prefix}_{resource}_{day}_{last_ts}")
            model.Add(last_after == 0)
            has_after[last_ts] = last_after

            for i in range(n - 2, -1, -1):
                ts = ordered_timeslots[i]
                next_ts = ordered_timeslots[i + 1]
                a = model.NewBoolVar(f"after_{prefix}_{resource}_{day}_{ts}")
                model.Add(a >= has_after[next_ts])
                model.Add(a >= occupancy[(resource, day, next_ts)])
                model.Add(a <= has_after[next_ts] + occupancy[(resource, day, next_ts)])
                has_after[ts] = a

            for ts in ordered_timeslots:
                occ = occupancy[(resource, day, ts)]
                idle = model.NewBoolVar(f"idle_{prefix}_{resource}_{day}_{ts}")
                model.Add(idle <= has_before[ts])
                model.Add(idle <= has_after[ts])
                model.Add(idle <= 1 - occ)
                model.Add(idle >= has_before[ts] + has_after[ts] + (1 - occ) - 2)
                idle_vars.append(idle)

    return idle_vars

# weiche Constraints für Optimierung
commission_occupancy = build_occupancy(2, config["commissions"], "commission")
room_occupancy = build_occupancy(3, config["rooms"], "room")

commission_idle_vars = add_internal_idle_penalties(
    commission_occupancy, config["commissions"], "commission"
 )
room_idle_vars = add_internal_idle_penalties(room_occupancy, config["rooms"], "room")

# Kommissions-Leerlauf wird stärker gewichtet ("insbesondere")
model.Minimize(2 * sum(commission_idle_vars) + sum(room_idle_vars))

solver = cp_model.CpSolver()
solver.parameters.max_time_in_seconds = 20.0
status = solver.Solve(model)

if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    solution_ortools = {}
    for p_idx, pres in enumerate(presentations):
        for s_idx, slot in enumerate(slots):
            if (p_idx, s_idx) in x and solver.Value(x[(p_idx, s_idx)]) == 1:
                solution_ortools[pres] = slot
                break
    solution_ortools = dict(sorted(solution_ortools.items(), key=lambda item: (item[0][0], item[0][1])))

    optimization_report = {
        "status": "OPTIMAL" if status == cp_model.OPTIMAL else "FEASIBLE",
        "objective_value": solver.ObjectiveValue(),
        "commission_idle_slots": sum(solver.Value(v) for v in commission_idle_vars),
        "room_idle_slots": sum(solver.Value(v) for v in room_idle_vars),
    }
else:
    solution_ortools = None
    optimization_report = None

{"solution": solution_ortools, "optimization": optimization_report}

{'solution': None, 'optimization': None}

## Ausführen und Ergebnisanalyse

Implementieren. Begründen. Testen. Interpretieren.

In [ ]:
## config A
{
    ("G5", "A"): ("Tue", "4", "K5", "L2"),
    ("G1", "A"): ("Wed", "4", "K5", "L2"),
    ("G3", "A"): ("Wed", "3", "K5", "L2"),
    ("G5", "B"): ("Wed", "4", "K4", "L3"),
    ("G4", "A"): ("Wed", "2", "K5", "L2"),
    ("G2", "A"): ("Wed", "4", "K3", "L1"),
    ("G2", "B"): ("Thu", "3", "K4", "L3"),
    ("G4", "B"): ("Thu", "1", "K4", "L3"),
    ("G3", "B"): ("Thu", "2", "K4", "L3"),
    ("G1", "B"): ("Thu", "4", "K4", "L3"),
    ("G5", "C"): ("Thu", "4", "K5", "L2"),
    ("G1", "C"): ("Fri", "4", "K5", "L2"),
    ("G2", "C"): ("Fri", "3", "K5", "L2"),
    ("G3", "C"): ("Fri", "2", "K5", "L2"),
    ("G4", "C"): ("Fri", "1", "K5", "L2"),
}


## config B
{
    ("G2", "A"): ("Wed", "4", "K1", "L1"),
    ("G1", "A"): ("Wed", "4", "K3", "L2"),
    ("G3", "A"): ("Wed", "3", "K3", "L2"),
    ("G1", "B"): ("Thu", "4", "K2", "L3"),
    ("G2", "B"): ("Thu", "3", "K2", "L3"),
    ("G3", "B"): ("Thu", "2", "K2", "L3"),
    ("G2", "C"): ("Fri", "4", "K1", "L2"),
    ("G1", "C"): ("Fri", "4", "K3", "L3"),
    ("G3", "C"): ("Fri", "3", "K3", "L3"),
}

## Constraint Validator
Implementieren. Prüfen. Begründen und interpretieren.

### Unmöglichkeit von Config C

Konfiguration C besitzt kein Modell, dass alle Constraints erfüllt. Dies lässt sich beweisen anhand der folgenden Gegebenheiten zeigen:
1. Alle Präsentationen müssen in der Reihenfolge A -> B -> C gehalten werden (Constraint)
2. Eine Gruppe kann nur eine Präsentation pro Tag halten (Constraint)
3. Eine Kommission kann nicht zwei Vorträge parallel betreuen (Constraint)
4. Kommission "K1" ist die einzige, die Präsentation B betreuen kann
5. Kommission "K3" ist die einzige, die Präsentation C betreuen kann
6. K3 ist am Freitag nicht verfügbar

Aus (3) und (4) ergibt sich, dass es gibt eine Gruppe `G` gibt, die ihre B-Präsentation im 9. Verfügbarkeits-Slot von Kommission `K1` halten muss. Dieser 9. Slot ist Slot 2 am Donnerstag. Aus (1) und (2) folgt, dass diese Gruppe `G` ihre C-Präsentation am Tag darauf halten muss, also am Freitag. Aus (5) und (6) folgt, dass es am Freitag keinen Slot mehr geben kann, an dem `G` präsentieren kann.

Dadurch ist bewiesen, dass dieser Datensatz keine Lösung besitzt.